# Goals

Determine how the bug described in https://github.com/IQSS/dataverse.harvard.edu/issues/500 can adversley affect search results of users of Harvard Dataverse and other Dataverse repositories.

# Plan

- Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

- For Harvard Dataverse:
    - Get the list of URLs and names of RORs in the externalvocabularyvalue table

    - Get the list of author affiliations with RORs; the list and count of datasets that have those author affiliations; and if each ROR is in the externalvocabularyvalue table
    - For each author affiliation with a ROR:
        - Search 1: Use the Search API to search for datasets by the name of the author affiliation
        - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
        - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

- For other Dataverse installations:
    - Determine which Dataverse installations have datasets with ROR URLs in their Author Affiliations or Funding Information Agencies
    - For each installation:

        - Get the list of author affiliations with RORs and the list and count of datasets that have those author affiliations (I don't know what's in each installation's externalvocabularyvalue table, so I can't include if each ROR is in the externalvocabularyvalue table)
        - For each author affiliation with a ROR:
            - Search 1: Use the Search API to search for datasets by the name of the author affiliation
            - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
            - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

# Import libraries and set variables

In [ ]:
import csv
import json
import os
import pandas as pd
from pathlib import Path
import platform
import re
import requests
import sys
from time import sleep
from tqdm import tqdm
tqdmBarFormat = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

# Collect affiliation names, RORs and other info

## Parse Dataverse JSON exports to get author affiliation metadata

In [ ]:
# Get the Affiliation Names and ROR URLs that depositors have entered in the Author Affiliation 
# fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

collectionRange = '2026.08.25-2026.09.02'
authorAffiliationCsvFilePath = os.path.join(
    'data', 'raw', f'author_affiliations_{collectionRange}.csv')

# Create CSV file and add headerrow
with open(authorAffiliationCsvFilePath, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'dataset_pid', 'latest_dataset_version_number',
        'expanded_value_key_present',
        'author_affiliation_name', 'author_affiliation_ror'])

# dataverseJsonExportsDirectory = os.path.join(desktopPath, 'testing')
dataverseJsonExportsDirectory = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/all_json_metadata_files'

jsonFilePaths = list(Path(dataverseJsonExportsDirectory).glob('*(latest_version).json'))
rorRegex = r'https?://ror\.org/0[a-hj-km-np-tv-z|0-9]{6}[0-9]{2}$'

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=jsonFilePaths)
for jsonFilePath in loopObj:
    jsonFileName = os.path.basename(jsonFilePath)
    loopObj.set_postfix_str(jsonFileName)
    with open(jsonFilePath, 'r', encoding='utf-8') as f:
        datasetMetadata = json.load(f)

        # Get names of metadata blocks and name of first metadata block whose name matches the give metadatablockName
        matchingMetadataBlockName = None
        metadataBlockNamesList = []
        metadatablockDict = common_functions.improved_get(datasetMetadata, 'data.metadataBlocks')
        for metadatablock in metadatablockDict:
            metadataBlockNamesList.append(metadatablock)

        metadatablockName = 'citation'
        if metadatablockName in metadataBlockNamesList:
            matchingMetadataBlockName = metadatablockName

        # If the exact metadata block name is not in the list, do a less exact search
        elif metadatablockName not in metadataBlockNamesList:
            matchingMetadataBlockNames = [s for idx, s in enumerate(metadataBlockNamesList) if metadatablockName in s]
            if matchingMetadataBlockNames:
                matchingMetadataBlockName = matchingMetadataBlockNames[0]
            else:
                matchingMetadataBlockName is None

        if matchingMetadataBlockName is not None:
            citationMetadataBlockDict = datasetMetadata['data']['metadataBlocks'][matchingMetadataBlockName]
            for field in citationMetadataBlockDict['fields']:
                if field['typeName'] == 'author':
                    authorsDict = field
                    for author in authorsDict['value']:
                        if 'authorAffiliation' in author:

                            # Get basic metadata
                            basicInfoDict = datasetMetadata['data']
                            keysToRemove = ['metadataBlocks', 'files']
                            basicInfoDict = {key: value for key, value in basicInfoDict.items() if key not in keysToRemove}

                            datasetPid = basicInfoDict['datasetPersistentId']

                            majorVersionNumber = basicInfoDict['versionNumber']
                            minorVersionNumber = basicInfoDict['versionMinorNumber']
                            datasetVersionNumber = f'{majorVersionNumber}.{minorVersionNumber}'

                            # Get Author affiliation metadata
                            authorAffiliationDict = author['authorAffiliation']
                            authorAffiliationValue = authorAffiliationDict['value']

                            if 'expandedvalue' in authorAffiliationDict:
                                expandedValueKeyPresent = True
                                authorAffiliationExpandedValue = authorAffiliationDict['expandedvalue']
                                authorAffiliationName = common_functions.improved_get(authorAffiliationExpandedValue, 'termName')
                                # Use rorRegex to verify authorAffiliationValue is a valid ROR URL
                                if re.match(rorRegex, authorAffiliationValue):
                                    authorAffiliationRor = authorAffiliationValue

                            elif 'expandedvalue' not in authorAffiliationDict:
                                expandedValueKeyPresent = False
                                authorAffiliationValue = authorAffiliationValue
                                # Use rorRegex to verify authorAffiliationValue is a valid ROR URL
                                if re.match(rorRegex, authorAffiliationValue):
                                    authorAffiliationName = None
                                    authorAffiliationRor = authorAffiliationValue
                                else:
                                    authorAffiliationName = authorAffiliationValue
                                    authorAffiliationRor = None

                            affiliationToAdd = [
                                datasetPid, datasetVersionNumber, expandedValueKeyPresent,
                                authorAffiliationName, authorAffiliationRor
                            ]

                            # Write values to CSV file
                            with open(authorAffiliationCsvFilePath, mode='a', newline='', encoding='utf-8-sig') as f:
                                writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                                writer.writerow(affiliationToAdd)
                            

## Reformat author affiliation info

In [ ]:
authorAffiliationCsvFilePath = os.path.join(
    'data', 'processed', 
    'author_affiliations_2026.08.25-2026.09.02.csv')

authorAffiliationsDf = pd.read_csv(
    authorAffiliationCsvFilePath,
    encoding='utf-8-sig')

print(f'Rows in authorAffiliationsDf: {len(authorAffiliationsDf)}')

distinctDatasetsCount = len(list(set(authorAffiliationsDf['dataset_pid'].to_list())))
print(f'Count of distinct dataset PIDs in authorAffiliationsDf: {distinctDatasetsCount}')

authorAffiliationsDf.head()

Rows in authorAffiliationsDf: 715519
Count of distinct dataset PIDs in authorAffiliationsDf: 308586


,dataset_pid,latest_dataset_version_number,expanded_value_key_present,author_affiliation_name,author_affiliation_ror
0,doi:10.17026/DANS-26A-CQ4R,2.0,False,RAAP Archeologisch Adviesbureau B.V.,NaN
1,hdl:2014/29246,2.0,False,UNKNOWN,NaN
2,hdl:2014/29246,2.0,False,UNKNOWN,NaN
3,hdl:2014/29246,2.0,False,UNKNOWN,NaN
4,doi:10.34810/DATA3521,1.0,True,Universitat Politècnica de Catalunya,https://ror.org/03mb6wj31


In [48]:
# To author_affiliations_2026.08.25-2026.09.02.csv, add columns to show the installation name and installation URL of each dataset.
# Use /Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataset_pids_from_most_known_dataverse_installations.csv

allInstallationMetadata2026Directory = os.path.join(
        '/Users/juliangautier/Documents',
        'all_installation_metadata_2026.08.25_22.57.59')

datasetsByInstallationsDf = pd.read_csv(
    os.path.join(
        allInstallationMetadata2026Directory,
        'dataset_pids_from_most_known_dataverse_installations.csv'),
    usecols=[
        'dataverse_installation_name',
        'dataverse_installation_url',
        'dataset_pid', 'dataset_pid_url'],
    dtype=str,
    encoding='utf-8-sig')
datasetsByInstallationsDf.head()

print(f'Rows in datasetsByInstallationsDf: {len(datasetsByInstallationsDf)}')

distinctInstallationsCount = len(list(set(datasetsByInstallationsDf['dataverse_installation_name'].to_list())))
print(f'Count of distinct installations in datasetsByInstallationsDf: {distinctInstallationsCount}')

datasetsByInstallationsDf.head()

Rows in datasetsByInstallationsDf: 541464
Count of distinct installations in datasetsByInstallationsDf: 127


,dataverse_installation_name,dataverse_installation_url,dataset_pid,dataset_pid_url
0,Recherche Data Gouv,https://entrepot.recherche.data.gouv.fr,doi:10.15454/H8CBYZ,https://doi.org/10.15454/H8CBYZ
1,Recherche Data Gouv,https://entrepot.recherche.data.gouv.fr,doi:10.15454/ZUKN90,https://doi.org/10.15454/ZUKN90
2,Recherche Data Gouv,https://entrepot.recherche.data.gouv.fr,doi:10.15454/QCKBKF,https://doi.org/10.15454/QCKBKF
3,Recherche Data Gouv,https://entrepot.recherche.data.gouv.fr,doi:10.17180/forest-biomass-fr-guiana-map-2016,https://doi.org/10.17180/forest-biomass-fr-gui...
4,Recherche Data Gouv,https://entrepot.recherche.data.gouv.fr,doi:10.17180/CANOPY-HEIGHT-FR-GUIANA-2014,https://doi.org/10.17180/CANOPY-HEIGHT-FR-GUIA...


In [54]:
# Merge two dataframes
authorAffiliationsDf2 = authorAffiliationsDf.merge(
    datasetsByInstallationsDf, how='left',
    on=['dataset_pid'])

authorAffiliationsDf2 = authorAffiliationsDf2[[
    'dataverse_installation_name', 'dataverse_installation_url',
    'dataset_pid', 'dataset_pid_url', 'latest_dataset_version_number',
    'expanded_value_key_present', 'author_affiliation_name', 'author_affiliation_ror'
]]

print(f'Rows in authorAffiliationsDf2: {len(authorAffiliationsDf2)}')

distinctInstallationsCount = len(list(set(authorAffiliationsDf2['dataverse_installation_name'].to_list())))
distinctDatasetsCount = len(list(set(authorAffiliationsDf2['dataset_pid'].to_list())))
print(f'Count of distinct installations in authorAffiliationsDf2: {distinctInstallationsCount}')
print(f'Count of distinct dataset PIDs in authorAffiliationsDf2: {distinctDatasetsCount}')

authorAffiliationsDf2.to_csv(
    os.path.join(desktopPath, 'authorAffiliationsDf2.csv'), 
    index=False, encoding='utf-8-sig')

authorAffiliationsDf2.head()

Rows in authorAffiliationsDf2: 715519
Count of distinct installations in authorAffiliationsDf2: 127
Count of distinct dataset PIDs in authorAffiliationsDf2: 308586


,dataverse_installation_name,dataverse_installation_url,dataset_pid,dataset_pid_url,latest_dataset_version_number,expanded_value_key_present,author_affiliation_name,author_affiliation_ror
0,DANS Data Station Archaeology,https://archaeology.datastations.nl,doi:10.17026/DANS-26A-CQ4R,https://doi.org/10.17026/DANS-26A-CQ4R,2.0,False,RAAP Archeologisch Adviesbureau B.V.,NaN
1,JPL Open Repository,https://dataverse.jpl.nasa.gov,hdl:2014/29246,https://hdl.handle.net/2014/29246,2.0,False,UNKNOWN,NaN
2,JPL Open Repository,https://dataverse.jpl.nasa.gov,hdl:2014/29246,https://hdl.handle.net/2014/29246,2.0,False,UNKNOWN,NaN
3,JPL Open Repository,https://dataverse.jpl.nasa.gov,hdl:2014/29246,https://hdl.handle.net/2014/29246,2.0,False,UNKNOWN,NaN
4,CORA. Research Data Repository (RDR),https://dataverse.csuc.cat,doi:10.34810/DATA3521,https://doi.org/10.34810/DATA3521,1.0,True,Universitat Politècnica de Catalunya,https://ror.org/03mb6wj31


## Data quality check of authorAffiliationsDf2

In [ ]:
# Data quality check: find dataset_pid values that appear more than once in datasetsByInstallationsDf
# and list the installation(s) each one is in

# keep=False marks every row of a duplicated dataset_pid, not just the later ones

duplicatePidRowsDf = authorAffiliationsDf2[
    authorAffiliationsDf2.duplicated(subset=['dataset_pid'], keep=False)]

# One row per duplicated dataset_pid, with the row count and the distinct installations it's in
duplicatePidsDf = (
    duplicatePidRowsDf
    .groupby('dataset_pid', as_index=False)
    .agg(
        row_count=('dataset_pid', 'size'),
        installation_count=('dataverse_installation_name', 'nunique'),
        installation_names=('dataverse_installation_name', lambda x: sorted(set(x.dropna()))),
        installation_urls=('dataverse_installation_url', lambda x: sorted(set(x.dropna()))))
    .sort_values('row_count', ascending=False))

# print(f'Rows with a dataset_pid that appears more than once: {len(duplicatePidRowsDf)}')
# print(f'Distinct duplicated dataset_pid values: {len(duplicatePidsDf)}')
print(f'Duplicated dataset_pid values in more than one installation: {(duplicatePidsDf["installation_count"] > 1).sum()}')
# print(f'Duplicated dataset_pid values repeated within one installation: {(duplicatePidsDf["installation_count"] == 1).sum()}')

duplicatePidsDf.to_csv(
    os.path.join(desktopPath, 'duplicate_dataset_pids_by_installation.csv'),
    index=False, encoding='utf-8-sig')

Rows with a dataset_pid that appears more than once: 552214
Distinct duplicated dataset_pid values: 145281
Duplicated dataset_pid values in more than one installation: 0
Duplicated dataset_pid values repeated within one installation: 145281


,dataset_pid,row_count,installation_count,installation_names,installation_urls
46789,doi:10.5683/SP4/KDEB13,225,1,[Borealis],[https://borealisdata.ca]
15130,doi:10.17617/3.7YLWMU,160,1,[Edmond],[https://edmond.mpg.de]
36041,doi:10.5683/SP2/63OGB0,156,1,[Borealis],[https://borealisdata.ca]
36783,doi:10.5683/SP2/OUUICF,115,1,[Borealis],[https://borealisdata.ca]
47376,doi:10.57745/9U1J85,107,1,[Recherche Data Gouv],[https://entrepot.recherche.data.gouv.fr]
32722,doi:10.34894/PX5IVZ,106,1,[DataverseNL],[https://dataverse.nl]
140330,perma:LIST.AQ7IVN,100,1,[LORE - LIST Open Repository],[https://lore.list.lu]
35061,doi:10.48732/F9HY-3J10,100,1,[Johns Hopkins Research Data Repository],[https://archive.data.jhu.edu]
139944,perma:LIST.7DE8R8,100,1,[LORE - LIST Open Repository],[https://lore.list.lu]
139952,perma:LIST.7FCRCC,100,1,[LORE - LIST Open Repository],[https://lore.list.lu]


## Get list of unique ROR URLs and their organization names
From author_affiliations_2026.08.25-2026.09.02.csv, get the unique ROR URLs and names of organizations that depositors have entered in the Author Affiliation fields.

In [ ]:
authorAffiliationCsvFilePath = os.path.join(
    'data', 'processed', 
    'author_affiliations_2026.08.25-2026.09.02.csv')

authorAffiliationRorsDf = pd.read_csv(
    authorAffiliationCsvFilePath,
    usecols=['author_affiliation_name', 'author_affiliation_ror'],
    dtype=str,
    encoding='utf-8-sig')

authorAffiliationRorsDf.head()

Rows in authorAffiliationRorsDf: 715519


,author_affiliation_name,author_affiliation_ror
0,RAAP Archeologisch Adviesbureau B.V.,NaN
1,UNKNOWN,NaN
2,UNKNOWN,NaN
3,UNKNOWN,NaN
4,Universitat Politècnica de Catalunya,https://ror.org/03mb6wj31


In [ ]:
# Data quality checks: missing values, duplicates, values in author_affiliation_ror column 
# that aren't valid ROR URLs

print(f'Rows: {len(authorAffiliationRorsDf)}')
print(f'Rows with no ROR URL: {authorAffiliationRorsDf["author_affiliation_ror"].isna().sum()}')
print(f'Rows with a ROR URL but no organization name: {(authorAffiliationRorsDf["author_affiliation_ror"].notna() & authorAffiliationRorsDf["author_affiliation_name"].isna()).sum()}')
print(f'Fully duplicated rows: {authorAffiliationRorsDf.duplicated().sum()}')

rorUrlsList = list(authorAffiliationRorsDf['author_affiliation_ror'].dropna())
rorUrlsList = list(set(rorUrlsList))
print(f'Distinct ROR URLs in rorUrlsList: {len(rorUrlsList)}')

nonMatchingRorUrls = [i for i in rorUrlsList if not re.match(rorRegex, i)]
print(f'Malformed ROR URLs: {len(nonMatchingRorUrls)}')

Rows: 715519
Rows with no ROR URL: 683442
Rows with a ROR URL but no organization name: 1552
Fully duplicated rows: 656889
Distinct items in rorUrlsList: 2221
Malformed ROR URLs: 0


In [ ]:
# Create a dataframe with one row per distinct ROR URL, a list of each distinct organization name for that ROR URL
# and the count of organization names. Missing names are skipped, so a ROR URL with no name in any row gets an empty list
authorAffiliationRorsDf = (
    authorAffiliationRorsDf
    .dropna(subset=['author_affiliation_ror'])
    .groupby('author_affiliation_ror', as_index=False)
    .agg(organization_names_from_expandedvalue_key=('author_affiliation_name', lambda x: sorted(set(x.dropna()))))
    .rename(columns={'author_affiliation_ror': 'ror_url'}))

authorAffiliationRorsDf['organization_name_count_from_expandedvalue_key'] = authorAffiliationRorsDf['organization_names_from_expandedvalue_key'].apply(len)
authorAffiliationRorsDf.head()

Distinct ROR URLs: 2221


In [ ]:
# Data quality checks: missing values, duplicates, values in author_affiliation_ror column 
# that aren't valid ROR URLs

print(f'Rows: {len(authorAffiliationRorsDf)}')
rorUrlsList = list(authorAffiliationRorsDf['ror_url'].dropna())
rorUrlsList = list(set(rorUrlsList))
print(f'Distinct ROR URLs in rorUrlsList: {len(rorUrlsList)}')

print(f'Rows with no ROR URL: {authorAffiliationRorsDf["ror_url"].isna().sum()}')
print(f'Rows with a ROR URL but no organization name: {(authorAffiliationRorsDf["organization_name_count_from_expandedvalue_key"] == 0).sum()}')

nonMatchingRorUrls = [i for i in rorUrlsList if not re.match(rorRegex, i)]
print(f'Malformed ROR URLs: {len(nonMatchingRorUrls)}')

# authorAffiliationRorsDf.to_csv(
#     os.path.join('data', 'processed', 'ror_urls_and_organization_names.csv'), 
#     index=False, encoding='utf-8-sig')

Rows: 2221
Rows with no ROR URL: 0
Rows with a ROR URL but no organization name: 505
Distinct ROR URLs in rorUrlsList: 2221
Malformed ROR URLs: 0


In [ ]:
# Add a column, 'display_names_2026.11.01', to authorAffiliationRorsWithDisplayNamesDf, 
# that shows the display names of each ROR URL in the ror_url column

displayNamesColumnName = 'ror_display_names_2026.11.01'

authorAffiliationRorsWithDisplayNamesDf = authorAffiliationRorsDf

# Get the display name of each distinct ROR URL from the ROR API (one request per ROR URL)
rorUrlsList = list(authorAffiliationRorsWithDisplayNamesDf['ror_url'].dropna().unique())
displayNamesDict = {}

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=rorUrlsList)
for rorUrl in loopObj:
    loopObj.set_postfix_str(rorUrl)
    rorInfoDict = common_functions.get_org_data_from_ror(rorUrl)
    displayNamesDict[rorUrl] = rorInfoDict['displayName']
    sleep(.25)

authorAffiliationRorsWithDisplayNamesDf[displayNamesColumnName] = authorAffiliationRorsWithDisplayNamesDf['ror_url'].map(displayNamesDict)

print(f'Rows with no display name: {authorAffiliationRorsWithDisplayNamesDf[displayNamesColumnName].isna().sum()}')

authorAffiliationRorsWithDisplayNamesDf.to_csv(
    os.path.join(desktopPath, 'ror_urls_and_organization_names.csv'),
    index=False, encoding='utf-8-sig')
authorAffiliationRorsWithDisplayNamesDf.head()

100%|██████████| 2221/2221 [Elapsed:25:39<Remaining:00:00 1.44it/s, https://ror.org/05yptqp13]

Rows with no display name: 0


,ror_url,organization_names_from_expandedvalue_key,organization_name_count_from_expandedvalue_key,display_names_2026.11.01
0,https://ror.org/00013q465,[Pontifical Catholic University of Peru],1,Pontificia Universidad Católica del Perú
1,https://ror.org/0004r6b85,[University of Applied Sciences Biberach],1,University of Applied Sciences Biberach
2,https://ror.org/0008bt423,[University of Fiji],1,University of Fiji
3,https://ror.org/0008xqs48,[Institut d'Investigació Biomédica de Bellvitge],1,Institut d'Investigació Biomédica de Bellvitge
4,https://ror.org/000917t60,[],0,Autonomous University of San Luis Potosí


# Create 
To authorAffiliationsDf, add ror_url_display_name column
To authorAffiliationsDf, add a column that shows the display name of the ROR URL when there is one, using authorAffiliationRorsWithDisplayNamesDf
Save new dataframe as authorAffiliationsDf3


# Split authorAffiliationsDf3 by installation name and installation url
Create new CSV files for each 


In [ ]:
dataframeList = [group.reset_index(drop=True) for _, group in df.groupby(['columnA', 'columnB'])]

# Get all names, ROR URLs in Author Affiliation and Funding Information Agency fields
Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields in datasets published in all known Dataverse installations. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

In [ ]:
# Set the directory of JSON files to parse (destinationDirectory is set in the copy cell above) 
# and the path of the CSV file to save
jsonDirectory = destinationDirectory

In [ ]:
# Parse each JSON file and collect the ROR URL and organization name of each author affiliation that has a ROR URL,
# along with the name of the file that has it
rorRows = []
filesWithRorsCount = 0

def findAuthorAffiliationRors(obj, rows, fileName):
    # Recursively search the metadata for authorAffiliation fields whose value is a ROR URL
    if isinstance(obj, dict):
        if obj.get('typeName') == 'authorAffiliation' and 'ror.org' in str(obj.get('value', '')):
            rows.append({
                'ror_url': obj['value'],
                'organization_name': obj.get('expandedvalue', {}).get('termName'),
                'file_name': fileName})
        for value in obj.values():
            findAuthorAffiliationRors(value, rows, fileName)
    elif isinstance(obj, list):
        for item in obj:
            findAuthorAffiliationRors(item, rows, fileName)

jsonFileNames = [f for f in os.listdir(jsonDirectory) if f.endswith('.json')]

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=jsonFileNames)
for jsonFileName in loopObj:
    loopObj.set_postfix_str(jsonFileName)
    with open(os.path.join(jsonDirectory, jsonFileName), 'r', encoding='utf-8') as f:
        datasetMetadata = json.load(f)

    rowCountBefore = len(rorRows)
    findAuthorAffiliationRors(datasetMetadata, rorRows, jsonFileName)
    if len(rorRows) > rowCountBefore:
        filesWithRorsCount += 1

rorsDf = pd.DataFrame(rorRows, columns=['ror_url', 'organization_name', 'file_name'])

In [ ]:
# Create one row per unique ROR URL. Include as lists the organization names and file names that have each ROR,
# the count of organization names, and the count of files that have each ROR
uniqueRorsDf = (
    rorsDf
    .groupby('ror_url')
    .agg(
        organization_names_from_export=('organization_name', lambda x: sorted(set(x.dropna()))),
        dataset_names=('file_name', lambda x: sorted(set(x))))
    .reset_index()
    .sort_values('ror_url'))
uniqueRorsDf['organization_name_count'] = uniqueRorsDf['organization_names_from_export'].apply(len)
uniqueRorsDf['dataset_count'] = uniqueRorsDf['dataset_names'].apply(len)

uniqueRorsDf.head()

In [ ]:
# For each ROR URL that has no organization name in uniqueRorsDf, get the organization name from the ROR API
# and attach to a new column, organization_names_from_ROR_API, in uniqueRorsDf

rorUrlsWithoutName = uniqueRorsDf.loc[
    uniqueRorsDf['organization_names_from_export'].apply(len) == 0, 'ror_url'].tolist()

# Dictionary of ROR URLs and the one organization name that the ROR API returns for each
rorApiNames = {}
failedRorUrls = []

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=rorUrlsWithoutName)
for rorUrl in loopObj:
    loopObj.set_postfix_str(f'{rorUrl}, failed: {len(failedRorUrls)}')
    try:
        rorInfoDict = common_functions.get_org_data_from_ror(rorUrl)
        rorApiName = rorInfoDict['displayName']
        rorApiNames[rorUrl] = rorApiName
    except requests.exceptions.RequestException:
        failedRorUrls.append(rorUrl)
    sleep(1)

# The new column is empty for ROR URLs that have a name in the export
uniqueRorsDf['organization_names_from_ROR_API'] = uniqueRorsDf['ror_url'].map(rorApiNames)

print(f'ROR URLs without a name in the export: {len(rorUrlsWithoutName)}; failed API requests: {len(failedRorUrls)}')

In [ ]:
# Save the ROR URLs, organization names and dataset names to a CSV file on my desktop
uniqueRorsDf = uniqueRorsDf[[
    'ror_url', 'organization_names_from_export', 'organization_name_count', 'organization_names_from_ROR_API', 'dataset_count', 'dataset_names'
]]

csvFilePath = os.path.join(desktopPath, 'author_affiliation_rors.csv')
uniqueRorsDf.to_csv(csvFilePath, index=False, encoding='utf-8-sig')

print(f'Saved {len(uniqueRorsDf)} unique ROR URLs to {csvFilePath}')
uniqueRorsDf.head()

In [ ]:
# Summarize the data in uniqueRorsDf (the data saved in author_affiliation_rors.csv). 
# Save the summary to a text file on my desktop
summaryFilePath = os.path.join(desktopPath, 'author_affiliation_rors_data_quality_checks.txt')

# ROR URLs that don't match the pattern https://ror.org/ followed by 9 alphanumeric characters
malformedRorUrlCount = (~uniqueRorsDf['ror_url'].str.fullmatch(r'https://ror\.org/[0-9a-z]{9}')).sum()

hasExportName = uniqueRorsDf['organization_name_count'] > 0
hasApiName = uniqueRorsDf['organization_names_from_ROR_API'].notna()

summaryLines = [
    f'Files parsed: {len(jsonFileNames)}',
    f'Files with at least one ROR in an author affiliation: {uniqueRorsDf["dataset_names"].explode().nunique()}',
    f'Unique ROR URLs: {len(uniqueRorsDf)}',
    f'Malformed ROR URLs: {malformedRorUrlCount}',
    f'ROR URLs with a name in the export: {hasExportName.sum()}',
    f'ROR URLs with more than one name in the export: {(uniqueRorsDf["organization_name_count"] > 1).sum()}',
    f'ROR URLs with no name in the export: {(~hasExportName).sum()}',
    f'ROR URLs with no name in the export that got a name from the ROR API: {(~hasExportName & hasApiName).sum()}',
    f'ROR URLs with no name in the export or from the ROR API: {(~hasExportName & ~hasApiName).sum()}',
    f'Failed ROR API requests: {len(failedRorUrls)}']

with open(summaryFilePath, mode='w', encoding='utf-8') as f:
    f.write('\n'.join(summaryLines) + '\n')

In [ ]:
# Load and query the Dataverse JSON exports of the latest version of each dataset. If the export includes ROR URLs in the author affiliation 
# field, save the dataset PID, the ROR URL, and ROR name if it's there

filePath = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataverse_json_metadata_from_each_known_dataverse_installation_unzipped/CORA._Research_Data_Repository_(RDR)_2026.09.02_11.05.39/Dataverse_JSON_metadata_2026.09.02_11.05.39/doi_10.34810_DATA3521_v1.0(latest_version).json'
with open(filePath, 'r') as f1:
    datasetMetadata = f1.read()
    datasetMetadata = json.loads(datasetMetadata)

citationBlock = datasetMetadata['data']['metadataBlocks']['citation']
for field in citationBlock['fields']:
    if field['typeName'] == 'author':
        authors = field['value']

for author in authors:
    authorAffiliationName = author['authorAffiliation']['expandedvalue']['termName']
    authorAffiliationRor = author['authorAffiliation']['value']

    print(f'{authorAffiliationName} ({authorAffiliationRor})')

# print(json.dumps(authors, indent=4))

authorsDf = pd.read_csv(
    'author(citation)_2026.08.25-2026.09.02.csv',
    # parse_dates=['dataset_publication_date'],
    usecols=lambda x: x in ['dataset_pid', 'dataset_pid_url', 'authorAffiliation'],
    # dtype={'latest_dataset_version_number': str},
    encoding='utf-8-sig'
)

renamedColumns = {'authorAffiliation': 'author_affiliation'}
authorsDf = authorsDf.rename(columns=renamedColumns)

# Create new dataframe with only authors where there's a ROR in the authorAffiliation field, using regex to match the ROR pattern (https://ror.org/ followed by 9 alphanumeric characters)
authorsWithRORAffiliationsDf = authorsDf[authorsDf['author_affiliation'].str.contains(r'https://ror\.org/[0-9a-zA-Z]{9}', regex=True, na=False)]

print(f'Count of rows in authorsWithRORAffiliationsDf: {len(authorsWithRORAffiliationsDf)}')

authorsWithRORAffiliationsDf.info()

Use the dataset PID and my dataset_pids_from_most_known_dataverse_installations.csv file to indicate which installation each dataset PID is in, so that I can group the PIDs by Dataverse installation

In [ ]:
objectColumns = [
    'dataverse_installation_name',
    'dataset_pid',
    'dataset_pid_url']
typeDict = {col: str for col in objectColumns}

datasetsByInstallationsDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataset_pids_from_most_known_dataverse_installations.csv',
    usecols=lambda x: x in ['dataverse_installation_name', 'dataset_pid', 'dataset_pid_url'],
    dtype=typeDict,
    encoding='utf-8-sig'
)

# Measure bug's affect on findability of datasets in Harvard Dataverse

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in Harvard Dataverse

# Measure bug's affect on findability of datasets in other Dataverse installations

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in each Dataverse installation